# 🧠 CNN-LSTM BISINDO — Preprocessing Pipeline

Notebook ini menjalankan **seluruh pipeline preprocessing** secara berurutan:

```
Raw Video
  ↓
[Step 1] Frame Extraction
         → Blur Filter → Resize → Uniform Sampling → Video Cutting
  ↓
[Step 2] Augmentation
         → Frame Transforms → Frame Sampling → Padding
  ↓
[Step 3] Dataloader Builder
         → Train / Val / Test Split → Simpan .npz
```

**Untuk menambah dataset baru**: edit `config/dataset_config.py` saja, tidak perlu ubah script lain.

---
## 📦 0. Install Dependencies

In [ ]:
# Install dependencies yang diperlukan
!pip install opencv-python-headless numpy tqdm matplotlib

---
## ⚙️ 1. Setup & Konfigurasi

In [ ]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ── Auto-resolve ROOT repo & tambahkan ke path ────────────────
# PreProcess.ipynb ada di: <ROOT>/PreProcessing/
# Jadi ROOT = 1 level ke atas dari notebook ini
NOTEBOOK_DIR = Path(".").resolve()
ROOT = NOTEBOOK_DIR.parent          # PreProcessing/ → ROOT
PREPROCESS_DIR = NOTEBOOK_DIR       # = ROOT/PreProcessing/

# Tambahkan PreProcessing/ ke sys.path agar bisa import Config & Scripts
if str(PREPROCESS_DIR) not in sys.path:
    sys.path.insert(0, str(PREPROCESS_DIR))

print(f"ROOT repo      : {ROOT}")
print(f"PreProcessing  : {PREPROCESS_DIR}")

from Config.dataset_config import DATASET_CONFIGS, PREPROCESSING_CONFIG
from Scripts.frame_extractor import process_dataset
from Scripts.augmentor import process_augmentation
from Scripts.dataloader_builder import build_dataloader, load_split

print("Import berhasil")
print(f"Dataset tersedia: {list(DATASET_CONFIGS.keys())}")


In [ ]:
# ──────────────────────────────────────────────────────────────
# PILIH DATASET DI SINI
# Ganti ke nama dataset lain jika mau proses dataset berbeda
# ──────────────────────────────────────────────────────────────
ACTIVE_DATASET = "WLBisindo"

dataset_cfg = DATASET_CONFIGS[ACTIVE_DATASET]
preprocess_cfg = PREPROCESSING_CONFIG

# Path
RAW_VIDEO_DIR  = dataset_cfg["raw_video_dir"]   # Sesuaikan dengan lokasi video kamu
OUTPUT_DIR     = dataset_cfg["output_dir"]

# Path intermediary
FRAME_MANIFEST = os.path.join(OUTPUT_DIR, "frame_manifest.json")
AUG_MANIFEST   = os.path.join(OUTPUT_DIR, "augmented_manifest.json")

print(f"\n📂 Dataset     : {ACTIVE_DATASET}")
print(f"📂 Raw video   : {RAW_VIDEO_DIR}")
print(f"📂 Output dir  : {OUTPUT_DIR}")
print(f"\n⚙️  Preprocessing config:")
print(f"   Target FPS  : {preprocess_cfg['frame']['target_fps']}")
print(f"   Resize      : {preprocess_cfg['frame']['resize']}")
print(f"   Max frames  : {preprocess_cfg['frame']['max_frames']}")
print(f"   Seq length  : {preprocess_cfg['sequence']['fixed_length']}")
print(f"   Aug enabled : {preprocess_cfg['augmentation']['enabled']}")
print(f"   Aug multiplier: {preprocess_cfg['augmentation']['augment_multiplier']}x")

---
## 🎬 2. Step 1 — Frame Extraction

**Input**: Raw `.mp4` videos  
**Proses**: Blur → Resize → Uniform Sampling → Cutting  
**Output**: `frames.npy` per video + `frame_manifest.json`

In [ ]:
# Cek apakah raw video directory ada
if not os.path.exists(RAW_VIDEO_DIR):
    print(f"⚠️  Direktori tidak ditemukan: {RAW_VIDEO_DIR}")
    print(f"\nSesuaikan path 'raw_video_dir' di config/dataset_config.py")
    print(f"Contoh: 'D:/Softwares/GithubProject/CNN-LSTM_BISINDO_Android/WLBisindo'")
else:
    from pathlib import Path
    video_count = len(list(Path(RAW_VIDEO_DIR).rglob("*.mp4")))
    print(f"✅ Direktori ditemukan!")
    print(f"   Total video .mp4: {video_count}")

In [ ]:
# ── Jalankan Frame Extraction ──────────────────────────────────
# overwrite=False → skip video yang sudah diproses (resume-able)
# overwrite=True  → proses ulang semuanya dari awal

frame_stats = process_dataset(
    raw_video_dir=RAW_VIDEO_DIR,
    output_dir=OUTPUT_DIR,
    dataset_config=dataset_cfg,
    preprocessing_config=preprocess_cfg,
    overwrite=False,
)

print(f"\n✅ Frame extraction selesai!")
print(f"   Diproses : {frame_stats['processed']}")
print(f"   Di-skip  : {frame_stats['skipped_short'] + frame_stats['skipped_error']}")

In [ ]:
# Visualisasi distribusi jumlah frame per video
if frame_stats.get("frame_counts"):
    counts = frame_stats["frame_counts"]
    plt.figure(figsize=(10, 4))
    plt.hist(counts, bins=30, color='steelblue', edgecolor='white')
    plt.axvline(np.mean(counts), color='red', linestyle='--', label=f'Mean: {np.mean(counts):.1f}')
    plt.axvline(preprocess_cfg['sequence']['fixed_length'], color='orange',
                linestyle='--', label=f"Fixed length: {preprocess_cfg['sequence']['fixed_length']}")
    plt.xlabel('Jumlah Frame per Video')
    plt.ylabel('Frekuensi')
    plt.title('Distribusi Frame Count setelah Extraction')
    plt.legend()
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 Statistik frame:")
    print(f"   Min  : {min(counts)}")
    print(f"   Max  : {max(counts)}")
    print(f"   Mean : {np.mean(counts):.1f}")
    print(f"   Std  : {np.std(counts):.1f}")

In [ ]:
# Preview: tampilkan beberapa frame dari video pertama
with open(FRAME_MANIFEST) as f:
    manifest = json.load(f)

if manifest:
    sample = manifest[0]
    frames = np.load(sample["frames_npy"])
    
    print(f"Sample: {sample['video_name']}")
    print(f"Label : {sample['label_name']} (idx={sample['label_idx']})")
    print(f"Frames: {frames.shape}  →  (N_frames, H, W, C)")
    
    # Tampilkan 8 frame
    n_show = min(8, len(frames))
    fig, axes = plt.subplots(1, n_show, figsize=(2.5*n_show, 3))
    for i, ax in enumerate(axes):
        idx = int(i * len(frames) / n_show)
        ax.imshow(frames[idx])
        ax.set_title(f'Frame {idx}', fontsize=8)
        ax.axis('off')
    plt.suptitle(f"Preview: {sample['video_name']}", fontsize=10)
    plt.tight_layout()
    plt.show()

---
## 🔄 3. Step 2 — Augmentation

**Input**: `frame_manifest.json`  
**Proses**: Frame Transforms + Padding ke fixed_length  
**Output**: `augmented/*.npy` + `augmented_manifest.json`

In [ ]:
augmented_manifest = process_augmentation(
    frame_manifest_path=FRAME_MANIFEST,
    output_dir=OUTPUT_DIR,
    preprocessing_config=preprocess_cfg,
    overwrite=False,
)

originals = [x for x in augmented_manifest if not x.get('is_augmented')]
augmented = [x for x in augmented_manifest if x.get('is_augmented')]

print(f"\n✅ Augmentasi selesai!")
print(f"   Original  : {len(originals)}")
print(f"   Augmented : {len(augmented)}")
print(f"   Total     : {len(augmented_manifest)}")

In [ ]:
# Preview: bandingkan original vs augmented
if augmented_manifest:
    orig_item = next(x for x in augmented_manifest if not x.get('is_augmented'))
    aug_item  = next(x for x in augmented_manifest if x.get('is_augmented') and
                     x['video_name'] == orig_item['video_name'])

    orig_frames = np.load(orig_item['augmented_npy'])
    aug_frames  = np.load(aug_item['augmented_npy'])

    fig, axes = plt.subplots(2, 6, figsize=(15, 5))
    for i in range(6):
        idx = int(i * preprocess_cfg['sequence']['fixed_length'] / 6)
        axes[0, i].imshow(orig_frames[idx])
        axes[0, i].set_title(f'Orig [{idx}]', fontsize=8)
        axes[0, i].axis('off')
        axes[1, i].imshow(aug_frames[idx])
        axes[1, i].set_title(f'Aug [{idx}]', fontsize=8)
        axes[1, i].axis('off')
    axes[0, 0].set_ylabel('Original', fontsize=10)
    axes[1, 0].set_ylabel('Augmented', fontsize=10)
    plt.suptitle(f"Augmentation Preview: {orig_item['video_name']}")
    plt.tight_layout()
    plt.show()

---
## 💾 4. Step 3 — Build Dataloader

**Input**: `augmented_manifest.json`  
**Proses**: Stratified split → pack ke .npz  
**Output**: `train.npz`, `val.npz`, `test.npz`, `label_map.json`

In [ ]:
split_info = build_dataloader(
    augmented_manifest_path=AUG_MANIFEST,
    output_dir=OUTPUT_DIR,
    preprocessing_config=preprocess_cfg,
    dataset_config=dataset_cfg,
    overwrite=False,
)

print(f"\n✅ Dataloader siap!")
for k, v in split_info.items():
    print(f"   {k}: {v}")

In [ ]:
# Verifikasi: load dan cek shape
X_train, y_train, meta_train = load_split(split_info['train_path'])
X_val,   y_val,   meta_val   = load_split(split_info['val_path'])
X_test,  y_test,  meta_test  = load_split(split_info['test_path'])

print("\n📦 Data shapes:")
print(f"   X_train : {X_train.shape}  →  (N, T, H, W, C)")
print(f"   y_train : {y_train.shape}")
print(f"   X_val   : {X_val.shape}")
print(f"   X_test  : {X_test.shape}")

# Load label map
with open(split_info['label_map_path']) as f:
    label_map = json.load(f)
print(f"\n🏷️  Jumlah kelas: {len(label_map)}")
print(f"   Contoh: { {k: label_map[k] for k in list(label_map.keys())[:5]} }")

In [ ]:
# Visualisasi distribusi kelas
from collections import Counter

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, (y, name) in zip(axes, [(y_train, 'Train'), (y_val, 'Val'), (y_test, 'Test')]):
    counts = Counter(y.tolist())
    labels_sorted = sorted(counts.keys())
    vals = [counts[l] for l in labels_sorted]
    ax.bar(labels_sorted, vals, color='steelblue')
    ax.set_title(f'{name} — {sum(vals)} samples')
    ax.set_xlabel('Label Index')
    ax.set_ylabel('Count')
plt.suptitle('Distribusi Kelas per Split')
plt.tight_layout()
plt.show()

---
## ✅ Ringkasan Preprocessing

Semua step selesai. File yang dihasilkan:

```
data/processed/WLBisindo/
├── frames/                    ← Step 1: frame per video (.npy)
├── augmented/                 ← Step 2: sequence augmented (.npy)
├── dataloader/
│   ├── train.npz              ← (N_train, T, H, W, C)
│   ├── val.npz
│   ├── test.npz
│   ├── label_map.json
│   └── split_info.json
├── frame_manifest.json        ← metadata Step 1
└── augmented_manifest.json    ← metadata Step 2
```

**Next step**: Buka `02_model_training.ipynb` untuk training CNN-LSTM + Attention.

In [ ]:
# ── Summary final ──────────────────────────────────────────────
print("="*55)
print(f"  PREPROCESSING SELESAI — {ACTIVE_DATASET}")
print("="*55)
print(f"  Kelas         : {split_info['num_classes']}")
print(f"  Sequence len  : {split_info['sequence_length']} frame")
print(f"  Frame size    : {split_info['frame_size']}")
print(f"  Train samples : {split_info['train_count']}")
print(f"  Val samples   : {split_info['val_count']}")
print(f"  Test samples  : {split_info['test_count']}")
print("="*55)